# premiere league data

In [2]:
import pandas as pd
epl_matches_loaded = pd.read_csv("data/epl_matches.csv")
display(epl_matches_loaded.head(40))
# list seasons and number of matches per season
season_counts = epl_matches_loaded['season'].value_counts().sort_index()
print("Number of matches per season:")
display(season_counts)


FileNotFoundError: [Errno 2] No such file or directory: 'data/epl_matches.csv'

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

if 'epl_matches_loaded' not in globals():
    candidate_paths = [
        Path('data/epl_matches.csv'),
        Path('notebooks/epl_matches.csv'),
        Path('../data/epl_matches.csv'),
    ]
    for csv_path in candidate_paths:
        if csv_path.exists():
            epl_matches_loaded = pd.read_csv(csv_path)
            break
    else:
        raise FileNotFoundError('Could not find epl_matches.csv in the expected locations.')

# Quick exploration of the Premier League matches dataset
print("Dataset shape:", epl_matches_loaded.shape)
print("\nColumns:")
print(epl_matches_loaded.columns.tolist())

print("\nMissing values per column:")
display(epl_matches_loaded.isna().sum().sort_values(ascending=False))

print("\nData types:")
display(epl_matches_loaded.dtypes)

if 'season' in epl_matches_loaded.columns:
    season_counts = epl_matches_loaded['season'].value_counts().sort_index()
    print("\nMatches per season:")
    display(season_counts)

if {'home_team', 'away_team', 'home_goals', 'away_goals'}.issubset(epl_matches_loaded.columns):
    explored = epl_matches_loaded.copy()
    explored['result'] = np.select(
        [explored['home_goals'] > explored['away_goals'], explored['home_goals'] < explored['away_goals']],
        ['Home win', 'Away win'],
        default='Draw'
    )
    explored['total_goals'] = explored['home_goals'] + explored['away_goals']

    print("\nResult distribution:")
    display(explored['result'].value_counts())

    print("\nMost frequent home teams:")
    display(explored['home_team'].value_counts().head(10))

    print("\nMost frequent away teams:")
    display(explored['away_team'].value_counts().head(10))

    print("\nHighest scoring matches:")
    display(explored.sort_values('total_goals', ascending=False)[['season', 'home_team', 'away_team', 'home_goals', 'away_goals', 'total_goals']].head(10))

    plt.figure(figsize=(10, 4))
    sns.countplot(data=explored, x='result', order=['Home win', 'Draw', 'Away win'])
    plt.title('Match Result Distribution')
    plt.xlabel('Result')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    sns.histplot(explored['total_goals'], bins=20, kde=True, color='steelblue')
    plt.title('Total Goals per Match')
    plt.xlabel('Total goals')
    plt.ylabel('Matches')
    plt.tight_layout()
    plt.show()

    team_stats = (
        explored.groupby('home_team')
        .agg(matches=('home_team', 'size'), avg_home_goals=('home_goals', 'mean'), avg_away_goals_allowed=('away_goals', 'mean'))
        .sort_values('matches', ascending=False)
        .head(10)
    )
    print("\nTop 10 home teams by match count:")
    display(team_stats)
else:
    numeric_cols = epl_matches_loaded.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        print("\nNumeric columns summary:")
        display(epl_matches_loaded[numeric_cols].describe().T)

    categorical_cols = epl_matches_loaded.select_dtypes(include=['object', 'category']).columns.tolist()
    if categorical_cols:
        print("\nTop categories in the first few categorical columns:")
        for col in categorical_cols[:5]:
            print(f"\n{col}:")
            display(epl_matches_loaded[col].value_counts().head(10))


Dataset shape: (3040, 7)

Columns:
['season', 'stage', 'date', 'home_team', 'away_team', 'home_team_goal', 'away_team_goal']

Missing values per column:


season            0
stage             0
date              0
home_team         0
away_team         0
home_team_goal    0
away_team_goal    0
dtype: int64


Data types:


season            object
stage              int64
date              object
home_team         object
away_team         object
home_team_goal     int64
away_team_goal     int64
dtype: object


Matches per season:


...

date
2012-05-13 00:00:00    10
2013-05-19 00:00:00    10
2014-05-11 00:00:00    10
2008-12-26 00:00:00    10
2010-05-09 00:00:00    10
2009-05-24 00:00:00    10
2015-12-26 00:00:00    10
2011-05-22 00:00:00    10
2015-01-01 00:00:00    10
2014-01-01 00:00:00    10
Name: count, dtype: int64


home_team:


home_team
Manchester United    152
Liverpool            152
Arsenal              152
Aston Villa          152
Tottenham Hotspur    152
Sunderland           152
Manchester City      152
Stoke City           152
Everton              152
Chelsea              152
Name: count, dtype: int64


away_team:


away_team
Arsenal              152
Everton              152
Stoke City           152
Aston Villa          152
Tottenham Hotspur    152
Manchester United    152
Manchester City      152
Sunderland           152
Liverpool            152
Chelsea              152
Name: count, dtype: int64